In [2]:
import sys
sys.path.append('../')
import scipy.io as sio
import mat73
import pandas as pd
import torch
import numpy as np
import torch.optim as optim
import torch.nn
import sklearn
import sklearn.metrics
import matplotlib.pyplot as plt
from alive_progress import alive_bar
from  utils.my_classes import dataset 
from torch.utils.data import DataLoader
import utils.DNN_functions as DNN_functions
import scipy
import random
import utils.AMSloss
from speechbrain.pretrained import SpeakerRecognition
import pickle
import os
from utils.my_classes import dataset
import utils.eval_metrics as eval_metrics
import copy
from speechbrain.pretrained import SpeakerRecognition

seed = 42  # You can choose any integer value as the seed
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
# Set seed for Python's random module
random.seed(seed)
# Set seed for NumPy
np.random.seed(seed)
# Set seed for PyTorch (CPU and GPU, if available)
if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
# Set deterministic flags for PyTorch (if available)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


In [3]:
#To get my GPU device - GTX 4070 :)
seed = 42  # You can choose any integer value as the seed
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu');

if torch.cuda.is_available():
    print(torch.cuda.device_count())
    print(torch.cuda.device(0))
    print(torch.cuda.get_device_name(0))
    print(device)

1
NVIDIA GeForce RTX 4070
cuda


In [4]:
models_folder = "ECAPA_TDNN/inference_models/"

data_path_male = "Data/not_normalize_all_features/male/"

data_path_female = "Data/not_normalize_all_features/female/"

In [ ]:
import torchaudio
from speechbrain.pretrained import EncoderClassifier

    
# load the model - the model that check if it's the sampe speaker or not
verification_model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir='./ECAPA_TDNN/pretrained_models/spkrec-ecapa-voxceleb',run_opts={"device":"cuda"} )

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir='./ECAPA_TDNN/pretrained_models/spkrec-ecapa-voxceleb',run_opts={"device":"cuda"})



In [5]:
Prior_target =  0.99

Prior_non_target = 1-Prior_target

cost_model_dcf = {
    'Cmiss_asv': 1,      # Cost of ASV falsely rejecting target
    'Cfa_asv': 10,       # Cost of ASV falsely accepting nontarget
}

# Load The Dataset:

In [1]:
import pickle
from ASV_utils.dcf_my_functions import dcf_formula

import pickle
results_list_dev_all = pickle.load(open("asv_results_dev_fixed_all.pkl", "rb"))
#results_list_dev_all = pickle.load(open("results_list_eval_all_include_gender_classification_1.pkl", "rb"))
gender_type = 'all'

target_scores_all_dev = results_list_dev_all.loc[(results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_all_dev =results_list_dev_all.loc[(results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

# min dcf one threshold for all Dataset:

In [6]:
#min dcf one threshold
asv_score_list = np.arange(0, 1.01, 0.001).tolist()
dcf_list = [];

for asv_score in asv_score_list:
    #print("current asv score is: ",asv_score)

    Pfa_asv = np.sum(nontarget_scores_all_dev >= asv_score) / nontarget_scores_all_dev.size
    Pmiss_asv = np.sum(target_scores_all_dev < asv_score) / target_scores_all_dev.size
    dcf = dcf_formula(Pfa_asv,Pmiss_asv,Prior_target = Prior_target,Prior_non_target= Prior_non_target,cost_model_dcf=cost_model_dcf,is_print=False)
    dcf_list.append(dcf)
    


print("target size:" ,target_scores_all_dev.size)
print("nontarget size:" ,nontarget_scores_all_dev.size)

print(np.argmin(dcf_list))  
print(np.min(dcf_list)) 
print(asv_score_list[np.argmin(dcf_list)])

target size: 1484
nontarget size: 5768
314
0.009307827179232203
0.314


# minDCF according to prediction Male / Female:

In [22]:
#minDCF prediction
asv_score_list = np.arange(0, 1.01, 0.001).tolist()
dcf_list = [];
gender_type = False
target_scores_male = results_list_dev_all.loc[(results_list_dev_all['gender_pred'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_male =results_list_dev_all.loc[(results_list_dev_all['gender_pred'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

for asv_score in asv_score_list:
    #print("current asv score is: ",asv_score)

    Pfa_asv = np.sum(nontarget_scores_male >= asv_score) / nontarget_scores_male.size
    Pmiss_asv = np.sum(target_scores_male < asv_score) / target_scores_male.size
    dcf = dcf_formula(Pfa_asv,Pmiss_asv,Prior_target = Prior_target,Prior_non_target= Prior_non_target,cost_model_dcf=cost_model_dcf,is_print=False)
    dcf_list.append(dcf)
    


print("target size:" ,target_scores_male.size)
print("nontarget size:" ,nontarget_scores_male.size)

print(np.argmin(dcf_list))  
print(np.min(dcf_list)) 
print(asv_score_list[np.argmin(dcf_list)])

target size: 924
nontarget size: 4536
330
0.008108465608465613
0.33


# minDCF according to wizard values Male / Female:

In [23]:
#minDCF wizard
asv_score_list = np.arange(0, 1.01, 0.001).tolist()
dcf_list = [];
gender_type = 'female'
target_scores_male = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_male =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

for asv_score in asv_score_list:
    #print("current asv score is: ",asv_score)

    Pfa_asv = np.sum(nontarget_scores_male >= asv_score) / nontarget_scores_male.size
    Pmiss_asv = np.sum(target_scores_male < asv_score) / target_scores_male.size
    dcf = dcf_formula(Pfa_asv,Pmiss_asv,Prior_target = Prior_target,Prior_non_target= Prior_non_target,cost_model_dcf=cost_model_dcf,is_print=False)
    dcf_list.append(dcf)
    


print("target size:" ,target_scores_male.size)
print("nontarget size:" ,nontarget_scores_male.size)

print(np.argmin(dcf_list))  
print(np.min(dcf_list)) 
print(asv_score_list[np.argmin(dcf_list)])

target size: 924
nontarget size: 4536
330
0.008108465608465613
0.33


# minDCF acorrding to thr of each set wizard/predicition : 

In [25]:
#total minDCF
gender_type = "female"
target_scores_female = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_female =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]


gender_type = "male"
target_scores_male = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_male =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]


asv_score_list = np.arange(0, 1.01, 0.001).tolist()
dcf_list = [];

asv_score_female_wizard = 0.33
asv_score_male_wizard = 0.247


#asv_score_female_pred = 0.33
#asv_score_male_pred = 0.247

#summing_female_nontar = np.sum(nontarget_scores_female >= asv_score_female_pred)
#summing_female_tar =  np.sum(target_scores_female < asv_score_female_pred)
#summing_male_nontar = np.sum(nontarget_scores_male >= asv_score_male_pred)
#summing_male_tar =  np.sum(target_scores_male < asv_score_male_pred)


summing_female_nontar = np.sum(nontarget_scores_female >= asv_score_female_wizard)
summing_female_tar =  np.sum(target_scores_female < asv_score_female_wizard)
summing_male_nontar = np.sum(nontarget_scores_male >= asv_score_male_wizard)
summing_male_tar =  np.sum(target_scores_male < asv_score_male_wizard)


Pfa_asv = (summing_female_nontar +  summing_male_nontar )/ (nontarget_scores_male.size+nontarget_scores_female.size)

Pmiss_asv = (summing_female_tar + summing_male_tar) / (target_scores_male.size+target_scores_female.size)
dcf = dcf_formula(Pfa_asv,Pmiss_asv,Prior_target = Prior_target,Prior_non_target= Prior_non_target,cost_model_dcf=cost_model_dcf,is_print=False)
dcf_list.append(dcf)



print(Pfa_asv)
print(Pmiss_asv)

print(np.argmin(dcf_list))  
print(np.min(dcf_list)) 


0.039875173370319004
0.0026954177897574125
0
0.006655980948891742


# EER with one threshold for all values :

In [15]:
#EER one threshold
import utils.my_functions as my_functions
import seaborn as sns
total_target = target_scores_all_dev

total_nontarget = nontarget_scores_all_dev

scores = np.concatenate((total_target,total_nontarget))

labels = np.concatenate((np.ones(len(total_target)),np.zeros(len(total_nontarget))))

eer, test_thresh = my_functions.compute_eer(labels,scores)

print("eer: ",eer*100)
print("test_thresh: ",test_thresh)

eer:  1.3477088948720872
test_thresh:  0.375164128713691


# Weighted Average EER of Male and Female - Wizard :

In [17]:
import utils.my_functions as my_functions

#pred gender
gender_type = "male"
target_scores_male = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_male =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

gender_type = "female"
target_scores_female = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_female =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]


scores_male = np.concatenate((target_scores_male,nontarget_scores_male))

labels_male = np.concatenate((np.ones(len(target_scores_male)),np.zeros(len(nontarget_scores_male))))

eer_male, test_thresh_male = my_functions.compute_eer(labels_male,scores_male)

print("eer: ",100*eer_male)
print("test_thresh: ",test_thresh_male)



scores_female = np.concatenate((target_scores_female,nontarget_scores_female))

labels_female = np.concatenate((np.ones(len(target_scores_female)),np.zeros(len(nontarget_scores_female))))

eer_female, test_thresh_female = my_functions.compute_eer(labels_female,scores_female)

print("eer: ",100*eer_female)
print("test_thresh: ",test_thresh_female)

print(((eer_female*scores_female.size + eer_male*scores_male.size)*100)/(scores_female.size+scores_male.size))

eer:  0.3571428571428544
test_thresh:  0.26685860835407926
eer:  1.0822510822512599
test_thresh:  0.39735834066532333
0.9030737602167502


# Weighted Average EER of Male and Female - Prediction :

In [18]:
#pred gender
gender_type = True
target_scores_male = results_list_dev_all.loc[(results_list_dev_all['gender_pred'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_male =results_list_dev_all.loc[(results_list_dev_all['gender_pred'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

gender_type = False
target_scores_female = results_list_dev_all.loc[(results_list_dev_all['gender_pred'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_female =results_list_dev_all.loc[(results_list_dev_all['gender_pred'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]


scores_male = np.concatenate((target_scores_male,nontarget_scores_male))

labels_male = np.concatenate((np.ones(len(target_scores_male)),np.zeros(len(nontarget_scores_male))))

eer_male, test_thresh_male = my_functions.compute_eer(labels_male,scores_male)

print("eer: ",eer_male)
print("test_thresh: ",test_thresh_male)



scores_female = np.concatenate((target_scores_female,nontarget_scores_female))

labels_female = np.concatenate((np.ones(len(target_scores_female)),np.zeros(len(nontarget_scores_female))))

eer_female, test_thresh_female = my_functions.compute_eer(labels_female,scores_female)

print("eer: ",eer_female)
print("test_thresh: ",test_thresh_female)

print(((eer_female*scores_female.size + eer_male*scores_male.size)*100)/(scores_female.size+scores_male.size))

eer:  0.003571428571428544
test_thresh:  0.26685860835407926
eer:  0.010822510822512598
test_thresh:  0.39735834066532333
0.9030737602167502


# EER of each set according to true labels:

In [19]:
#True gender
gender_type = "male"
target_scores_male = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_male =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

gender_type = "female"
target_scores_female = results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "target")]["pred_scores"]
nontarget_scores_female =results_list_dev_all.loc[(results_list_dev_all['gender'] == gender_type) & (results_list_dev_all['label_ground_truth'] == "nontarget")]["pred_scores"]

total_target = target_scores_male

total_nontarget = nontarget_scores_male

scores = np.concatenate((total_target,total_nontarget))

labels = np.concatenate((np.ones(len(total_target)),np.zeros(len(total_nontarget))))

eer, test_thresh = my_functions.compute_eer(labels,scores)

print("eer: ",eer)
print("test_thresh: ",test_thresh)


eer:  0.003571428571428544
test_thresh:  0.26685860835407926
